In [ ]:
import os
os.environ["HUGGINGFACE_API"] = ""
os.environ["GIT_TOKEN"] = ""

In [2]:
import os
from huggingface_hub import login


huggingface_api = os.environ["HUGGINGFACE_API"]
git_token = os.environ["GIT_TOKEN"]

if huggingface_api is None:
    raise RuntimeError("❌ Missing HUGGINGFACE_API in .env")

login(token=huggingface_api)


In [3]:
!git clone https://{git_token}@github.com/BGKhanh/Reasoning-Techniques-on-LLM.git

Cloning into 'Reasoning-Techniques-on-LLM'...
remote: Enumerating objects: 2473, done.
remote: Counting objects: 100% (391/391), done.
remote: Compressing objects: 100% (247/247), done.
remote: Total 2473 (delta 212), reused 292 (delta 127), pack-reused 2082 (from 1)
Receiving objects: 100% (2473/2473), 3.20 MiB | 14.76 MiB/s, done.
Resolving deltas: 100% (1659/1659), done.


In [4]:
%cd /Reasoning-Techniques-on-LLM

/Reasoning-Techniques-on-LLM


In [5]:
!git pull

Already up to date.


In [6]:
import sys
!{sys.executable} -m pip install -Uqr requirements.txt

In [7]:
!export FLASH_ATTENTION_SKIP_CUDA_BUILD=TRUE

In [ ]:
import sys
!{sys.executable} -m pip install -q https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.9.4/flash_attn-2.8.3+cu130torch2.11-cp312-cp312-linux_x86_64.whl

In [9]:
%cd lm-evaluation-harness
import sys
!{sys.executable} -m pip install -qe .

/Reasoning-Techniques-on-LLM/lm-evaluation-harness


# GEPA Optimization for Vietnamese SSA

Tối ưu prompt **Chain-of-Thought** cho task Structured Sentiment Analysis tiếng Việt
bằng `dspy.GEPA` (Genetic-Pareto reflective prompt optimizer, [arxiv:2507.19457](https://arxiv.org/abs/2507.19457)).

**Pipeline:**
1. Load 3 splits (train/dev/test) từ `data/vitoed_new/`
2. Định nghĩa `dspy.Signature` với system prompt làm docstring (zero-shot)
3. Wrap thành `dspy.ChainOfThought` module
4. Baseline eval trên valset
5. `dspy.GEPA.compile()` với feedback metric (decompose 6 SemEval-2022 F1)
6. So sánh trước/sau optimization trên test set

**Lưu ý:** Notebook này chỉ tối ưu cho **Chain-of-Thought**.
Các technique khác (Few-shot, Re-reading, Plan-and-Solve) giữ nguyên ở `src/prompt_templates/`
và chạy qua `lm-eval` riêng.

In [ ]:
import os
import sys
import json
import random
from pathlib import Path
from typing import Any, Dict, List, Optional

import dspy

# Resolve project root robustly: walk up từ cwd để tìm dir chứa `src/prompt_templates`.
# Robust với mọi cwd: notebook/ (local), project root (CI), lm-evaluation-harness/ (vastai cd).
def _find_project_root(start: Path, max_up: int = 5) -> Path:
    cur = start.resolve()
    for _ in range(max_up):
        if (cur / "src" / "prompt_templates").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start  

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Project-local helpers (zero side effects on import)
from src.prompt_templates.shared import SYSTEM_PROMPTS
from src.utils.postprocessing import extract_json_from_response, postprocess_response
from semeval22_structured_sentiment.evaluation.evaluate import (
    convert_opinion_to_tuple,
    sent_tuples_in_list,
    calculate_all_metrics,  # 6 SemEval-2022 F1 — single source of truth
)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"dspy version = {getattr(dspy, '__version__', 'unknown')}")

PROJECT_ROOT = /Reasoning-Techniques-on-LLM
dspy version = 3.2.1


## ⚙️ Configuration

Single source-of-truth: chỉnh **cell Configuration** (block `TECHNIQUE` / `DATA_DIR` / models) rồi run-all để swap model / budget / dataset size.
Các cell khác đều derive từ block này.

In [ ]:
# ===================================================================
# Technique (notebook này chỉ hỗ trợ Chain-of-Thought)
# ===================================================================
TECHNIQUE = "cot"
LANGUAGE = "vi"           # 'vi' | 'en' (system prompt pick từ shared.SYSTEM_PROMPTS)

# ===================================================================
# Data
# ===================================================================
DATA_DIR = PROJECT_ROOT / "data" / "vitoed_new"
TRAINSET_SIZE = 5000
VALSET_SIZE = 300
TESTSET_SIZE = None       # None = toàn bộ test split
SEED = 42

# ===================================================================
# Models
# Student: local vLLM (OpenAI-compatible). Tránh LiteLLM → Hugging Face Inference API (401 nếu thiếu key).
# Reflection: model mạnh (GPT-5/Claude Opus/Gemini-2.5-Pro) — chất lượng prompt phụ thuộc nặng vào đây.
# ===================================================================
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

# vLLM: `vllm serve` đăng ký model theo `--model` (mặc định tên = repo HF). LiteLLM cần prefix `openai/`.
VLLM_MODEL_ID = os.getenv("VLLM_MODEL_ID", "google/gemma-3-4b-it")
VLLM_PORT = int(os.getenv("VLLM_PORT", "8000"))
STUDENT_BASE_URL = os.getenv("STUDENT_BASE_URL", f"http://127.0.0.1:{VLLM_PORT}/v1")
STUDENT_API_KEY = os.getenv("STUDENT_API_KEY", "EMPTY")  # vLLM không bắt buộc; LiteLLM vẫn cần chuỗi
STUDENT_MODEL = f"openai/{VLLM_MODEL_ID}"

# REFLECTION_MODEL = "gemini/gemma-4-31b-it"   # WARN: 'gemini/' prefix dành cho Gemini family — đổi thành 'gemini/gemini-2.5-pro' hoặc 'vertex_ai/google/gemma-3-27b-it'
REFLECTION_MODEL = "gemini/gemini-3.1-flash-lite" 
STUDENT_MAX_TOKENS = 8196
REFLECTION_MAX_TOKENS = 64000

# ===================================================================
# GEPA budget
# ===================================================================
AUTO_BUDGET = "heavy"      # 'light' | 'medium' | 'heavy'
NUM_THREADS = 8
REFLECTION_MINIBATCH_SIZE = 20

# ===================================================================
# Output
# ===================================================================
OUTPUT_DIR = PROJECT_ROOT / "results" / "gepa" / TECHNIQUE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GEPA_LOG_DIR = OUTPUT_DIR / "log"
GEPA_LOG_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
print(f"OUTPUT_DIR   = {OUTPUT_DIR}")
print(f"GEPA_LOG_DIR = {GEPA_LOG_DIR}")

OUTPUT_DIR   = /Reasoning-Techniques-on-LLM/results/gepa/cot
GEPA_LOG_DIR = /Reasoning-Techniques-on-LLM/results/gepa/cot/log


## 📦 Data Pipeline

Convert raw VLSP samples → `dspy.Example` với `inputs = (text, sent_id)`.
Field `opinions` giữ nguyên format SemEval (offsets) để metric tái sử dụng đúng logic
`convert_opinion_to_tuple` từ `semeval22_structured_sentiment/evaluation/evaluate.py`.

In [70]:
def load_examples(path: Path) -> List[dspy.Example]:
    """Load 1 split JSON về `List[dspy.Example]`.

    Args:
        path: JSON file. Mỗi sample phải có 'sent_id', 'text', 'opinions'.

    Returns:
        List of `dspy.Example` với `with_inputs("text", "sent_id")`.
    """
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        raw = json.load(f)

    examples: List[dspy.Example] = []
    for s in raw:
        ex = dspy.Example(
            text=s["text"],
            sent_id=str(s["sent_id"]),
            opinions=s.get("opinions", []),  # gold tuples for metric
        ).with_inputs("text", "sent_id")
        examples.append(ex)
    return examples


train_all = load_examples(DATA_DIR / "train.json")
dev_all   = load_examples(DATA_DIR / "dev.json")
test_all  = load_examples(DATA_DIR / "test.json")

print(f"Loaded: train={len(train_all)}, dev={len(dev_all)}, test={len(test_all)}")
print("\nSample (first train ex):")
print(f"  text     = {train_all[0].text[:100]!r}")
print(f"  sent_id  = {train_all[0].sent_id}")
print(f"  opinions = {len(train_all[0].opinions)} opinion(s)")

GEPA Optimization:  11%|█         | 257/2435 [1:21:18<11:29:06, 18.98s/rollouts]

Loaded: train=7706, dev=1085, test=2194

Sample (first train ex):
  text     = 'quay quay cái lồn , thấy bị bắt nạt thì ra nói một câu bảo vệ người ta , có khi tối về lại có người '
  sent_id  = 1833
  opinions = 5 opinion(s)


In [71]:
def _shuffle_take(items: List[dspy.Example], n: Optional[int], seed: int) -> List[dspy.Example]:
    """Deterministic shuffle + take n; n=None → take all."""
    if n is None or n >= len(items):
        return list(items)
    rng = random.Random(seed)
    pool = list(items)
    rng.shuffle(pool)
    return pool[:n]


trainset = _shuffle_take(train_all, TRAINSET_SIZE, SEED)
valset   = _shuffle_take(dev_all,   VALSET_SIZE,   SEED + 1)
testset  = _shuffle_take(test_all,  TESTSET_SIZE,  SEED + 2)


def _polarity_dist(exs: List[dspy.Example]):
    dist: Dict[str, int] = {}
    total_ops = 0
    for ex in exs:
        for op in ex.opinions:
            pol = op.get("Polarity", "Unknown")
            dist[pol] = dist.get(pol, 0) + 1
            total_ops += 1
    return dist, total_ops


print(f"{'Split':<8s} | {'#examples':>10s} | {'#opinions':>10s} | distribution")
print("-" * 80)
for name, exs in [("trainset", trainset), ("valset", valset), ("testset", testset)]:
    dist, total = _polarity_dist(exs)
    print(f"{name:<8s} | {len(exs):>10d} | {total:>10d} | {dist}")

Split    |  #examples |  #opinions | distribution
--------------------------------------------------------------------------------
trainset |       2000 |       3926 | {'Neutral': 1038, 'Positive': 1231, 'Negative': 1657}
valset   |        200 |        378 | {'Neutral': 90, 'Positive': 112, 'Negative': 176}
testset  |       2194 |       4133 | {'Negative': 1681, 'Neutral': 1366, 'Positive': 1086}


## ✍️ DSPy Signatures — Seed Instructions for GEPA

Docstring của `Signature` chính là **thứ GEPA evolve** thông qua reflection LM
(xem [docs](https://dspy.ai/api/optimizers/GEPA/overview/)).

Theo quyết định: nạp **toàn bộ system prompt** (định nghĩa SOURCE/TARGET/POLAR_EXPRESSION/
POLARITY/INTENSITY) làm docstring; user prompt **zero-shot** (chỉ gồm `text` + `sent_id`)
để instruction sinh ra generalize tốt và dùng được lại cho các technique khác.

In [ ]:
import dspy
from typing import Dict

class SSACoTSignature(dspy.Signature):
    # CHUỖI DOCSTRING NÀY CHÍNH LÀ INSTRUCTIONS MÀ GEPA SẼ TỐI ƯU
    __doc__ = f"""{SYSTEM_PROMPTS[LANGUAGE].strip()}"""

    text = dspy.InputField(desc="Câu/bình luận tiếng Việt cần phân tích.")
    sent_id = dspy.InputField(desc="Mã định danh của câu.")
    
    output_json = dspy.OutputField(desc="Chuỗi JSON hợp lệ chứa kết quả trích xuất.")

print("=" * 80)
print(f"Initial signature instructions length = {len(SSACoTSignature.instructions)} chars")
print("=" * 80)
print(SSACoTSignature.instructions[:1500])
if len(SSACoTSignature.instructions) > 1500:
    print("...")

## 🧩 DSPy Modules

`dspy.ChainOfThought(SSACoTSignature)` tự động chèn predictor sinh **reasoning** trước
**output_json**, đúng tinh thần Chain-of-Thought.

Khi GEPA `compile()`, nó evolve `module.predict.predict.signature.instructions`
(tức docstring đã set ở Cell 8).

In [ ]:
class CoTSSAModule(dspy.Module):
    """Chain-of-Thought wrapper cho task SSA tiếng Việt.

    `forward(text, sent_id)` → object có `.output_json` (chuỗi JSON) và `.reasoning`
    (do `ChainOfThought` tự thêm). Metric sẽ parse output_json qua
    `extract_json_from_response` + `postprocess_response` để chuẩn hoá về SemEval format.
    """

    def __init__(self):
        super().__init__()
        self.predict = dspy.ChainOfThought(SSACoTSignature)

    def forward(self, text: str, sent_id: str):
        return self.predict(text=text, sent_id=str(sent_id))

In [74]:
# Structural smoke test (chưa cần LM — forward() sẽ test trong baseline eval ở Cell 21)
module = CoTSSAModule()

inner_sig = module.predict.predict.signature
assert "Polar_expression" in inner_sig.instructions, "System prompt không vào docstring!"
assert "text" in inner_sig.input_fields, "Missing 'text' input field"
assert "sent_id" in inner_sig.input_fields, "Missing 'sent_id' input field"
assert "output_json" in inner_sig.output_fields, "Missing 'output_json' output field"
# ChainOfThought tự thêm 'reasoning' field
assert "reasoning" in inner_sig.output_fields, "ChainOfThought không thêm 'reasoning'"

print("Module OK ✓")
print(f"  Predictor type     = {module.predict.__class__.__name__}")
print(f"  Instructions chars = {len(inner_sig.instructions)}")
print(f"  Input fields       = {list(inner_sig.input_fields.keys())}")
print(f"  Output fields      = {list(inner_sig.output_fields.keys())}")

Module OK ✓
  Predictor type     = ChainOfThought
  Instructions chars = 1828
  Input fields       = ['text', 'sent_id']
  Output fields      = ['reasoning', 'output_json']


## 📊 Metric & Feedback Functions

GEPA's metric protocol: `(gold, pred, trace, pred_name, pred_trace) -> float | dspy.Prediction`
([docs](https://dspy.ai/api/optimizers/GEPA/overview/#implementing-feedback-metrics)).

- Khi `pred_name is None` (dùng cho `dspy.Evaluate` baseline) → trả `float`.
- Khi có `pred_name` (GEPA gọi để xin feedback) → trả `dspy.Prediction(score, feedback)`.

**Score = SF1 (Sentiment Graph F1)** — chính metric của SemEval-2022.

**Feedback** decompose 6 sub-metrics + liệt kê *missed/spurious tuples* + show *gold opinions*
để reflection LM có ground truth khi đề xuất prompt mới (theo recipe "decompose outcomes" +
"ground in checks" trong GEPA paper).

In [75]:
def _parse_pred_to_sample(pred_obj, gold_text: str, gold_sent_id: str) -> dict:
    """Parse `pred.output_json` → SemEval sample format `{sent_id, text, opinions}`.

    Reuses `extract_json_from_response` + `postprocess_response` từ src.utils.postprocessing
    (tự fix các lỗi JSON phổ biến: backslash dư, trailing quote, unicode quote, etc.).
    """
    raw = getattr(pred_obj, "output_json", None) or "{}"
    if not isinstance(raw, str):
        raw = str(raw)
    extracted = extract_json_from_response(raw)
    normalized_str = postprocess_response(extracted, gold_text, gold_sent_id)
    try:
        return json.loads(normalized_str)
    except json.JSONDecodeError:
        return {"sent_id": gold_sent_id, "text": gold_text, "opinions": []}


def _normalize_metric_keys(metrics: Dict[str, float]) -> Dict[str, float]:
    """Map `calculate_all_metrics` keys ("Holder F1" → "Holder_F1") cho consistency."""
    return {k.replace(" ", "_"): float(v) for k, v in metrics.items()}


def _compute_per_doc_scores(gold_sample: dict, pred_sample: dict) -> Dict[str, Any]:
    """Tính 6 SemEval-2022 F1 cho 1 sample (per-doc, không phải corpus-level).

    Reuses `calculate_all_metrics` từ `semeval22_structured_sentiment/evaluation/evaluate.py`
    bằng cách gọi với dict 1 phần tử (cùng key cho gold + pred).

    Returns dict gồm 6 F1 (SF1, NSF1, Holder_F1, Target_F1, Exp_F1, Targeted_F1)
    + `_meta` (missed/spurious tuples cho feedback).
    """
    gt = convert_opinion_to_tuple(gold_sample)
    pt = convert_opinion_to_tuple(pred_sample)
    sid = str(gold_sample["sent_id"])

    # Single source of truth — same path as evaluate_single_dataset.py
    out: Dict[str, Any] = _normalize_metric_keys(
        calculate_all_metrics({sid: gt}, {sid: pt})
    )

    # Diagnostic: missed (FN) / spurious (FP) tuples — không có trong calculate_all_metrics
    missed_gold = [
        g for g in gt
        if not sent_tuples_in_list(g, pt, keep_polarity=True, mode="all")
    ]
    spurious_pred = [
        p for p in pt
        if not sent_tuples_in_list(p, gt, keep_polarity=True, mode="all")
    ]

    out["_meta"] = {
        "n_gold":        len(gt),
        "n_pred":        len(pt),
        "n_matched":     len(gt) - len(missed_gold),
        "missed_gold":   missed_gold,
        "spurious_pred": spurious_pred,
    }
    return out

In [ ]:
def _format_tuple_short(t) -> str:
    """Display 1 SemEval tuple compactly: (holder_idxs, target_idxs, exp_idxs, polarity)."""
    holder, target, exp, pol = t
    return f"(holder={sorted(holder)}, target={sorted(target)}, expr={sorted(exp)}, pol={pol})"


def _build_feedback(scores: Dict[str, Any], gold_sample: dict, pred_sample: dict) -> str:
    """Compose textual feedback decomposed by SemEval-2022 components.
    
    [UNRESTRICTED VERSION - High Rate Limit]
    Includes: Full original text, all metrics, exhaustive missed/spurious tuples, 
    full gold opinions, full predicted opinions, and targeted hints.
    """
    sf1   = scores["SF1"]
    nsf1  = scores["NSF1"]
    h_f1  = scores["Holder_F1"]
    t_f1  = scores["Target_F1"]
    e_f1  = scores["Exp_F1"]
    tg_f1 = scores["Targeted_F1"]
    meta  = scores["_meta"]

    lines = [
        "==================================================",
        "EVALUATION REPORT FOR CURRENT INSTRUCTION",
        "==================================================",
        f"\n[ORIGINAL INPUT TEXT]",
        f"\"{gold_sample['text']}\"\n",
        "[METRICS SUMMARY]",
        f"SemEval-2022 SF1 = {sf1:.3f}",
        f"Component F1 — Holder={h_f1:.3f}, Target={t_f1:.3f}, Expression={e_f1:.3f}",
        f"Strict F1 — Targeted={tg_f1:.3f}, NSF1 (no polarity)={nsf1:.3f}",
        f"Tuples Count — Gold={meta['n_gold']}, Predicted={meta['n_pred']}, Matched={meta['n_matched']}.\n"
    ]

    # 1. LIỆT KÊ TOÀN BỘ LỖI SAI (KHÔNG GIỚI HẠN)
    if meta["missed_gold"]:
        lines.append("[FALSE NEGATIVES - You missed these entire tuples or extracted them incorrectly]:")
        # In ra tất cả, không dùng [:5] nữa
        lines.extend(f"  - {_format_tuple_short(t)}" for t in meta["missed_gold"])
        
    if meta["spurious_pred"]:
        lines.append("\n[FALSE POSITIVES - You hallucinated these or extracted boundaries wrongly]:")
        # In ra tất cả
        lines.extend(f"  - {_format_tuple_short(t)}" for t in meta["spurious_pred"])

    # 2. CUNG CẤP CẢ BẢN CHUẨN LẪN BẢN LỖI (Full JSON)
    lines.append("\n[RAW DATA COMPARISON]")
    
    gold_json = json.dumps(gold_sample.get("opinions", []), ensure_ascii=False, indent=2)
    lines.append("Gold Label (The correct answer):")
    lines.append(gold_json)
    
    # Thêm bản Dự đoán để Teacher LM so sánh trực tiếp
    pred_json = json.dumps(pred_sample.get("opinions", []), ensure_ascii=False, indent=2)
    lines.append("\nYour Prediction (The flawed output):")
    lines.append(pred_json)

    # 3. BẮT BỆNH VÀ KÊ ĐƠN (Diagnostic Hints)
    lines.append("\n[DIAGNOSTIC HINTS FOR SYSTEM PROMPT IMPROVEMENT]")

    if meta["n_pred"] == 0 and meta["n_gold"] > 0:
        lines.append(
            "- CRITICAL ERROR (ZERO EXTRACTION): You failed to extract any opinions, but the text contains sentiment. "
            "Update instructions to strictly enforce extraction of ANY sentiment, "
            "including teencode, slang, emojis, or subtle sarcasm."
        )

    if meta["n_pred"] > 0 and sf1 == 0.0:
        lines.append(
            "- CHAIN-BREAK ERROR: SF1 is 0.0 despite having predictions. "
            "SemEval SF1 requires Holder, Target, AND Expression to overlap simultaneously with the Gold label. "
            "One or more of your predicted components is completely off. Fix the weakest component."
        )

    if h_f1 < 0.3 and meta["n_gold"] > 0:
        lines.append(
            f"- HOLDER HALLUCINATION: Holder_F1 is critically low ({h_f1:.3f}). "
            "In Vietnamese, the sentiment holder is often omitted (pro-drop). "
            "Instruct the model: 'If the subject is not explicitly stated in the text, leave Source/Holder EMPTY. DO NOT infer or invent pronouns like tôi, khách hàng'."
        )

    component_f1 = {"Holder": h_f1, "Target": t_f1, "Expression": e_f1}
    weakest_name = min(component_f1, key=lambda k: component_f1[k])
    weakest_score = component_f1[weakest_name]
    
    if 0.0 < weakest_score < 0.7:
        lines.append(
            f"- BOUNDARY ERROR: '{weakest_name}' span detection is weak ({weakest_score:.3f}). "
            "Update instructions: 'Spans MUST be EXACT surface substrings of the input text. "
            "Do not include surrounding articles/prepositions (e.g., 'cái', 'sự', 'những'). Do not fix typos. Extract the minimal necessary phrase.'"
        )

    if t_f1 > 0.5 and tg_f1 < 0.2:
        lines.append(
            f"- TARGETED STRICT FAILURE: Target overlap is okay ({t_f1:.3f}) but strict match is poor ({tg_f1:.3f}). "
            "Targeted_F1 requires EXACT token match for the Target. "
            "Instruct the model to be laser-focused when defining the boundaries of the evaluated object/aspect."
        )

    if nsf1 - sf1 >= 0.15: 
        lines.append(
            f"- POLARITY MISMATCH: NSF1 ({nsf1:.3f}) >> SF1 ({sf1:.3f}). "
            "Spans are mostly correct, but the POLARITY is misclassified. "
            "Update instructions to better define Positive/Negative/Neutral in Vietnamese context "
            "(e.g., handling sarcasm, idiomatic expressions, or double negatives)."
        )

    return "\n".join(lines)

In [ ]:
def ssa_metric_with_feedback(
    gold: dspy.Example,
    pred: dspy.Prediction,
    trace=None,
    pred_name: Optional[str] = None,
    pred_trace=None,
):
    """GEPA feedback metric.

    Args:
        gold: dspy.Example với (text, sent_id, opinions).
        pred: dspy.Prediction với .output_json (chuỗi JSON do CoT model sinh).
        trace, pred_name, pred_trace: GEPA-only kwargs.

    Returns:
        - float (SF1) khi `pred_name is None` (dùng cho dspy.Evaluate baseline).
        - dspy.Prediction(score, feedback) khi GEPA gọi để xin feedback.
    """
    gold_sample = {
        "sent_id":  str(gold.sent_id),
        "text":     gold.text,
        "opinions": gold.opinions,
    }

    try:
        pred_sample = _parse_pred_to_sample(pred, gold.text, str(gold.sent_id))
    except Exception as e:
        if pred_name is not None:
            return dspy.Prediction(
                score=0.0,
                feedback=(
                    f"Output failed to parse as JSON ({type(e).__name__}: {e}). "
                    "You MUST return a valid JSON object matching the exact schema. "
                    "No prose outside the JSON. Spans must be exact substrings of the input text."
                ),
            )
        elif trace is not None:
            return False 
        else:
            return 0.0

    try:
        scores = _compute_per_doc_scores(gold_sample, pred_sample)
    except Exception as e:
        if pred_name is not None:
            return dspy.Prediction(
                score=0.0, 
                feedback=(
                    f"JSON parsed successfully, but the semantic structure is invalid: {str(e)}. "
                    "Ensure Source, Target, and Polar_expression are strictly Lists of Strings. "
                    "If implicit/hidden, use an empty list []."
                )
            )
        elif trace is not None:
            return False
        else:
            return 0.0

    score = float(scores["SF1"])
    if pred_name is not None:
        feedback = _build_feedback(scores, gold_sample, pred_sample)
        return dspy.Prediction(score=score, feedback=feedback)

    if trace is not None:
        return score >= 1.0 

    return score

In [78]:
# Verify protocol with fake predictions (không cần LM)
class _FakePred:
    def __init__(self, output_json: str):
        self.output_json = output_json


_g = trainset[0]

# (1) Perfect prediction = gold
_perfect_json = json.dumps({
    "sent_id":  _g.sent_id,
    "text":     _g.text,
    "opinions": _g.opinions,
}, ensure_ascii=False)
_p_good = _FakePred(_perfect_json)

# (2) Empty prediction
_p_empty = _FakePred('{"opinions": []}')

# (3) Garbage
_p_bad = _FakePred("not json at all { { {")

# Test 1: pred_name=None → float
s1 = ssa_metric_with_feedback(_g, _p_good)
assert isinstance(s1, float), f"Expected float, got {type(s1)}"
print(f"[float, perfect-pred ]  SF1 = {s1:.3f}  (expected ~ 1.0)")

s2 = ssa_metric_with_feedback(_g, _p_empty)
print(f"[float, empty-pred   ]  SF1 = {s2:.3f}  (expected 0.0)")

# Test 2: pred_name=str → dspy.Prediction
out_p = ssa_metric_with_feedback(_g, _p_empty, pred_name="predict.predict")
assert isinstance(out_p, dspy.Prediction)
assert hasattr(out_p, "score") and hasattr(out_p, "feedback")
print(f"\n[Prediction, empty   ]  score = {out_p.score:.3f}")
print("Feedback preview (first 500 chars):")
print("-" * 80)
print(out_p.feedback[:500])
print("-" * 80)

# Test 3: bad JSON
out_b = ssa_metric_with_feedback(_g, _p_bad, pred_name="predict.predict")
print(f"\n[Prediction, bad json]  score = {out_b.score:.3f}")
print(f"Feedback (first 200): {out_b.feedback[:200]}")

[float, perfect-pred ]  SF1 = 1.000  (expected ~ 1.0)
[float, empty-pred   ]  SF1 = 0.000  (expected 0.0)

[Prediction, empty   ]  score = 0.000
Feedback preview (first 500 chars):
--------------------------------------------------------------------------------
SemEval-2022 SF1 = 0.000 (target ≥ 0.5).
Component F1 — Holder=0.000, Target=0.000, Expression=0.000, Targeted-strict=0.000; NSF1 (no polarity)=0.000.
Tuples — gold=2, predicted=0, matched=0.
Missed gold tuples (false negatives, top 5):
  - (holder=[], target=[], expr=[9, 10, 11, 12, 13], pol=Neutral)
  - (holder=[], target=[0, 1], expr=[2, 3, 4, 5, 6, 7], pol=Positive)
Gold opinions (for reference):
[
  {
    "Source": [
      [],
      []
    ],
    "Target": [
      [],
      []
    ],
    "P
--------------------------------------------------------------------------------

[Prediction, bad json]  score = 0.000
Feedback (first 200): SemEval-2022 SF1 = 0.000 (target ≥ 0.5).
Component F1 — Holder=0.000, Target=0.000, Expression=

## 🤖 Model Setup

- **vLLM (student server)**: cell ngay dưới — khởi chạy API cục bộ; cần GPU + `vllm` trong môi trường kernel.
- **Student LM**: `dspy.LM` trỏ tới `STUDENT_BASE_URL` (OpenAI-compatible) — inference train/val/test.
- **Reflection LM**: model đề xuất prompt mới — nên mạnh (paper khuyến nghị GPT-5/Opus, `temperature=1.0`).

In [84]:
# Khởi chạy vLLM OpenAI server cục bộ cho student LM (chạy sau cell Configuration).
# Tùy VRAM có thể thêm vào _vllm_cmd: --max-model-len 4096 --gpu-memory-utilization 0.9 --dtype bfloat16
import subprocess
import sys
import time
import urllib.error
import urllib.request

if "_vllm_proc" in globals() and _vllm_proc is not None and _vllm_proc.poll() is None:
    _vllm_proc.terminate()
    try:
        _vllm_proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        _vllm_proc.kill()

_vllm_env = os.environ.copy()
_hf = (
    _vllm_env.get("HUGGINGFACE_API_KEY")
    or _vllm_env.get("HF_TOKEN")
    or _vllm_env.get("HUGGINGFACE_HUB_TOKEN")
    or _vllm_env.get("HUGGINGFACE_API", "")
)
if _hf:
    _vllm_env.setdefault("HF_TOKEN", _hf)
    _vllm_env.setdefault("HUGGINGFACE_HUB_TOKEN", _hf)

_vllm_cmd = [
    sys.executable,
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--model",
    VLLM_MODEL_ID,
    "--host",
    "127.0.0.1",
    "--port",
    str(VLLM_PORT),
    "--dtype",
    "bfloat16"
]

_vllm_log = OUTPUT_DIR / "vllm_server.log"
_flog = open(_vllm_log, "a", encoding="utf-8", buffering=1)
_flog.write(f"\n\n==== vLLM start {time.strftime('%Y-%m-%d %H:%M:%S')} ====\n")
_flog.write(" ".join(_vllm_cmd) + "\n")
_flog.flush()

print("Starting vLLM — log:", _vllm_log)
_vllm_proc = subprocess.Popen(
    _vllm_cmd,
    env=_vllm_env,
    stdout=_flog,
    stderr=subprocess.STDOUT,
    cwd=str(PROJECT_ROOT),
)

_health = STUDENT_BASE_URL.rstrip("/") + "/models"
_deadline = time.time() + float(os.getenv("VLLM_START_TIMEOUT_S", "1200"))
_last_err = None
while time.time() < _deadline:
    if _vllm_proc.poll() is not None:
        _flog.close()
        tail = _vllm_log.read_text(encoding="utf-8", errors="replace")[-4000:]
        raise RuntimeError(
            f"vLLM exited early (code={_vllm_proc.returncode}). See {_vllm_log}. Tail:\n{tail}"
        )
    try:
        with urllib.request.urlopen(_health, timeout=5) as r:
            if r.status == 200:
                print(f"vLLM ready: {_health}")
                break
    except (urllib.error.URLError, TimeoutError, OSError) as e:
        _last_err = e
    time.sleep(2.0)
else:
    if _vllm_proc.poll() is None:
        _vllm_proc.terminate()
    _flog.close()
    raise RuntimeError(f"vLLM did not become ready in time. Last error: {_last_err}. Log: {_vllm_log}")

vLLM ready: http://127.0.0.1:8000/v1/models


In [ ]:
student_lm = dspy.LM(
    STUDENT_MODEL,
    max_tokens=STUDENT_MAX_TOKENS,
    temperature=0.0,
    api_base=STUDENT_BASE_URL,
    api_key=STUDENT_API_KEY,
    cache=True,
)
dspy.configure(lm=student_lm, track_usage=True)
print(f"Student LM configured: {STUDENT_MODEL} @ {STUDENT_BASE_URL}")

Student LM configured: openai/google/gemma-3-4b-it @ http://127.0.0.1:8000/v1


In [86]:
_reflection_kwargs = dict(
    model=REFLECTION_MODEL,
    temperature=1.0,
    max_tokens=REFLECTION_MAX_TOKENS,
    max_retries=5,
    timeout=300,
)
if GEMINI_API_KEY:
    _reflection_kwargs["api_key"] = GEMINI_API_KEY
elif "gemini/" in REFLECTION_MODEL:
    print("WARNING: GEMINI_API_KEY env var not set — LiteLLM sẽ thử default credential chain.")

reflection_lm = dspy.LM(**_reflection_kwargs)
print(f"Reflection LM configured: {REFLECTION_MODEL}")

Reflection LM configured: gemini/gemini-3.1-flash-lite


## 🚀 GEPA Optimization

- `auto={light, medium, heavy}` quyết định **rollout budget** (paper khuyến nghị `medium`/`heavy` cho production).
- `log_dir`: lưu state để **resume** nếu interrupt giữa chừng.
- `track_stats=True`: lưu `detailed_results` (Pareto front, val aggregate scores) cho phần Inspect.
- `reflection_lm`: model mạnh đề xuất prompt mới (cấu hình trong section **Model Setup**).

In [ ]:
optimizer = dspy.GEPA(
    metric=ssa_metric_with_feedback,
    auto=AUTO_BUDGET,
    reflection_minibatch_size=REFLECTION_MINIBATCH_SIZE,
    candidate_selection_strategy="pareto", 
    component_selector="all",              
    use_merge=True,                        
    max_merge_invocations=5,
    skip_perfect_score=True,
    num_threads=NUM_THREADS,
    track_stats=True,
    log_dir=str(GEPA_LOG_DIR),
    reflection_lm=reflection_lm,
    seed=SEED,
)

print(f"Starting GEPA: auto={AUTO_BUDGET}, "
      f"trainset={len(trainset)}, valset={len(valset)}, "
      f"reflection_lm={REFLECTION_MODEL}")
print(f"Logs: {GEPA_LOG_DIR}\n")

optimized_module = optimizer.compile(
    student=module,
    trainset=trainset,
    valset=valset,
)

print("\n✓ GEPA compile finished.")

In [ ]:
save_path = OUTPUT_DIR / "optimized_cot.json"
optimized_module.save(str(save_path))
print(f"Saved optimized module to: {save_path}")

opt_instr = optimized_module.predict.predict.signature.instructions
print(f"\nOptimized instruction length = {len(opt_instr)} chars")
print("=" * 80)
print(opt_instr)

## 🔍 Inspect Optimized Prompts

So sánh **baseline instruction** (system prompt gốc) vs **optimized instruction**
(GEPA evolved). Đồng thời in `detailed_results` (Pareto front, total metric calls)
để hiểu chi phí và quá trình tối ưu.

In [67]:
base_instr = SSACoTSignature.instructions
opt_instr  = optimized_module.predict.predict.signature.instructions

print("=" * 80)
print("BASELINE instruction (truncated to 1500):")
print("=" * 80)
print(base_instr[:1500])
print("\n" + "=" * 80)
print("OPTIMIZED instruction (truncated to 1500):")
print("=" * 80)
print(opt_instr[:1500])

# Length diff
print("\n" + "-" * 80)
print(f"Baseline length  = {len(base_instr)} chars, {len(base_instr.split())} words")
print(f"Optimized length = {len(opt_instr)} chars, {len(opt_instr.split())} words")

# detailed_results (chỉ có khi track_stats=True)
res = getattr(optimized_module, "detailed_results", None)
if res is not None:
    print("\n" + "=" * 80)
    print("GEPA detailed_results:")
    print("=" * 80)
    for attr in ("total_metric_calls", "num_full_val_evals"):
        val = getattr(res, attr, None)
        if val is not None:
            print(f"  {attr:<25s} = {val}")
    val_scores = getattr(res, "val_aggregate_scores", None)
    if val_scores is not None:
        try:
            print(f"  val_aggregate_scores      = {[round(float(x), 4) for x in val_scores]}")
        except Exception:
            print(f"  val_aggregate_scores      = {val_scores}")
    log_dir = getattr(res, "log_dir", None)
    if log_dir:
        print(f"  log_dir                   = {log_dir}")
else:
    print("\n(No detailed_results — track_stats=False?)")

BASELINE instruction (truncated to 1500):
Bạn là chuyên gia trong lĩnh vực phân tích cảm xúc tiếng Việt có cấu trúc. 
Nhiệm vụ của bạn là phân tích bình luận mạng xã hội và trích xuất các thành phần cảm xúc theo cấu trúc JSON.

ĐỊNH NGHĨA CÁC THÀNH PHẦN:

1. SOURCE (Nguồn gốc bình luận):
   - Người phát biểu ý kiến, có thể là người bình luận hoặc được trích dẫn
   - Thường là các đại từ nhân xưng: "Tôi", "Tao", "Mình", "Bọn tao", "Mẹ tui"
   - Có thể có hoặc không có trong câu

2. TARGET (Đối tượng hướng tới):
   - Cá nhân, tập thể, sự vật, hiện tượng mà bình luận hướng đến
   - Thường là các đại từ xưng hô: "Mày", "Cậu", "Anh ấy", "Bạn"
   - Có thể có hoặc không có trong câu

3. POLAR_EXPRESSION (Biểu thức cảm xúc):
   - Từ/cụm từ bày tỏ cảm xúc, ý nghĩ, cảm nhận, hành động
   - Bao gồm: tính từ cảm xúc, thán từ, hành động xúc phạm/khen ngợi
   - Ví dụ: "buồn", "vui", "tức giận", "đáng đời", "đánh"
   - BẮT BUỘC phải có

4. POLARITY (Tính chất cảm xúc):
   - Positive: Khích lệ, động v